## Model summary

Common notation: $r_t$ = log-return (in %), $\sigma_t^2$ = conditional variance, $x_t$ = exogenous variable (`tone` or `log_art_growth`), $z_t \sim t^*_\nu$ (standardized Student-t, $\nu > 2$), with $r_t = \sigma_t z_t$. We use `log_art_growth` (= $\log(1+N_t) - \log(1+N_{t-1})$, std $\approx 0.69$, range $[-4.2, +5.1]$) instead of raw `art_growth` (max $= 61$ on 2015-10-23) because the squared term $\gamma\, x_{t-1}^2$ in GARCH-X would otherwise be dominated by a single outlier day.

**1. GARCH(1,1) — Student-t baseline**
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2$$

**2. GARCH-X (tone)** — G&P Eq. (4), $x_{t-1} = \text{tone}_{t-1}$
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, x_{t-1}^2$$

**3. GARCH-X (log article growth)** — G&P Eq. (4), $x_{t-1} = \text{log\_art\_growth}_{t-1}$
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, x_{t-1}^2$$

**4. GARCHAND (tone)** — G&P Eq. (5), asymmetry driven by the sign of tone
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\,(1 + d_1\,\theta)\, x_{t-1}^2,\qquad d_1 = \mathbb{1}\{x_{t-1} < 0\},\ \theta > 0$$

**5. GARCHAND (log article growth)** — G&P Eq. (6), news effect active only when (log-)volume grows
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, d_2\, x_{t-1}^2,\qquad d_2 = \mathbb{1}\{x_{t-1} > 0\},\ \gamma > 0$$

**6. GARCHND (tone / log article growth)** — G&P Eq. (7), news effect active only in a high-volatility regime
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, d_3\, x_{t-1}^2,\qquad d_3 = \mathbb{1}\{\sigma_{t-1}^2 \geq \kappa\}$$
where $\kappa$ is calibrated from an annualized volatility threshold ($30\%$ and $50\%$): $\kappa = (v_{\text{ann}}/\sqrt{252})^2$. In the implementation $d_3$ is approximated by a smooth sigmoid $1/(1+e^{-K(\sigma_{t-1}^2 - \kappa)})$ with $K = 20$ so the log-likelihood is differentiable.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline


In [2]:
IN_PATH = '../CLEANED DATA/REMX_prices_sentiment_combined.xlsx'

df = pd.read_excel(IN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
df['r'] = np.log(df['Close'] / df['Close'].shift(1))
df['r'] = df['r'] - df['r'].mean()
r = df['r'].dropna()
r2 = (r ** 2)


# 1. GARCH(1,1) -- Student-t baseline


In [3]:
%%capture cap_garch
from arch import arch_model
from scipy.stats import norm

# 1. Returns in percent (standard scaling for GARCH stability)
r_pct = (df['r'].dropna() * 100.0)

# 2. Specify and fit GARCH(1,1) with constant mean and Student-t innovations
am = arch_model(r_pct, mean='Zero', vol='GARCH', p=1, q=1, dist='t')
res = am.fit(disp='off', cov_type='robust')

print(res.summary())
print()

# 3. Coefficient-level diagnostics (estimate / std err / t-ratio / p-value)
CRIT_10PCT = norm.ppf(1 - 0.10 / 2)
CRIT_5PCT  = norm.ppf(1 - 0.05 / 2)
CRIT_1PCT  = norm.ppf(1 - 0.01 / 2)

tbl = pd.DataFrame({
    'estimate':   res.params,
    'std_err':    res.std_err,
    't_ratio':    res.tvalues,
    'p_value':    res.pvalues,
})
tbl['significant_10pct'] = tbl['t_ratio'].abs() > CRIT_10PCT
tbl['significant_5pct']  = tbl['t_ratio'].abs() > CRIT_5PCT
tbl['significant_1pct']  = tbl['t_ratio'].abs() > CRIT_1PCT

print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print()

# 4. Model-level criteria
print(f'Log-likelihood : {res.loglikelihood: .4f}')
print(f'AIC            : {res.aic: .4f}')
print(f'BIC            : {res.bic: .4f}')
print()

# 5. Stationarity / persistence + tail thickness
alpha_hat = res.params.get('alpha[1]', float('nan'))
beta_hat  = res.params.get('beta[1]',  float('nan'))
nu_hat    = res.params.get('nu',       float('nan'))
persist   = alpha_hat + beta_hat
print(f'alpha[1] + beta[1] = {persist:.4f}   '
      f'({"stationary (<1)" if persist < 1 else "non-stationary (>=1)"})')
print(f'nu  = {nu_hat:.4f}   (smaller -> heavier tails; Gaussian as nu -> infinity)')
print()

# 6. Plain-language verdict
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<10}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')


# 2. GARCH-X (tone) -- tone matters?


In [4]:
%%capture cap_garchx_tone

# --- Shared MLE machinery for the five G&P (2019) GARCH variants ------------
# Estimation by Maximum Likelihood with Student-t innovations z_t ~ t*_nu
# (standardized: mean 0, variance 1) and zero-mean returns r_t = sigma_t z_t.
# Robust sandwich standard errors V_robust = H^{-1} J H^{-1}, with J the BHHH
# outer product of finite-difference scores (Bollerslev & Wooldridge, 1992).
# Returns are scaled to percent. The exogenous variables (tone, log_art_growth)
# are kept on their native scale: tone std ≈ 1.09 and log_art_growth std ≈ 0.69
# are already well-conditioned for γ·x² estimation.
from scipy.optimize import minimize
from scipy.special  import gammaln
from scipy.stats    import norm

from statsmodels.tools.numdiff import approx_hess

CRIT_10 = norm.ppf(1 - 0.10 / 2)
CRIT_5  = norm.ppf(1 - 0.05 / 2)
CRIT_1  = norm.ppf(1 - 0.01 / 2)

def _flag(t):
    a = abs(t)
    if a > CRIT_1:  return 'SIGNIFICANT at 1% (and 5%, 10%)'
    if a > CRIT_5:  return 'SIGNIFICANT at 5% (and 10%) only'
    if a > CRIT_10: return 'SIGNIFICANT at 10% only'
    return 'not significant'

def _sandwich_se(theta, neg_ll_fn, per_obs_fn, args):
    H = approx_hess(theta, neg_ll_fn, args=args)
    V_hess = np.linalg.inv(H)
    eps_fd = 1e-5
    T_obs  = len(args[0])
    G = np.empty((T_obs, len(theta)))
    for i in range(len(theta)):
        up = theta.copy(); up[i] += eps_fd
        dn = theta.copy(); dn[i] -= eps_fd
        ll_up, _ = per_obs_fn(up, *args)
        ll_dn, _ = per_obs_fn(dn, *args)
        G[:, i] = (ll_up - ll_dn) / (2 * eps_fd)
    J = G.T @ G
    V = V_hess @ J @ V_hess
    return np.sqrt(np.diag(V))

def _report(param_names, theta, se, ll, T_obs, persist_alpha_beta=None,
            extra_notes=None):
    t_ratios = theta / se
    p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_ratios)))
    tbl = pd.DataFrame({
        'estimate': theta, 'std_err': se,
        't_ratio': t_ratios, 'p_value': p_values,
    }, index=param_names)
    tbl['significant_10pct'] = tbl['t_ratio'].abs() > CRIT_10
    tbl['significant_5pct']  = tbl['t_ratio'].abs() > CRIT_5
    tbl['significant_1pct']  = tbl['t_ratio'].abs() > CRIT_1
    k   = len(theta)
    aic = -2.0 * ll + 2.0 * k
    bic = -2.0 * ll + k * np.log(T_obs)
    print()
    print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
    print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
    print()
    print(f'Log-likelihood : {ll: .4f}')
    print(f'AIC            : {aic: .4f}')
    print(f'BIC            : {bic: .4f}')
    if persist_alpha_beta is not None:
        ab = persist_alpha_beta
        print(f'alpha + beta   : {ab: .4f}   '
              f'({"stationary (<1)" if ab < 1 else "non-stationary (>=1)"})')
    if extra_notes:
        for line in extra_notes:
            print(line)
    print()
    print('Verdict:')
    for name, row in tbl.iterrows():
        print(f'  {name:<12}  t = {row["t_ratio"]:+.2f}   '
              f'p = {row["p_value"]:.4f}   -> {_flag(row["t_ratio"])}')
    return tbl

# Standardized Student-t per-observation log-density: z_t = r_t / sigma_t,
# with E[z] = 0 and Var[z] = 1 (requires nu > 2).
def _t_logpdf(z, nu):
    const = (gammaln((nu + 1.0) / 2.0) - gammaln(nu / 2.0)
             - 0.5 * np.log(np.pi * (nu - 2.0)))
    return const - 0.5 * (nu + 1.0) * np.log(1.0 + z**2 / (nu - 2.0))


# === GARCH-X (tone)  --  G&P Eq. (4), Student-t innovations ================
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1} + gamma tone^2_{t-1}
print('==== GARCH-X (tone)  --  G&P Eq. (4) with x = Tone, Student-t shocks ====')

r_pct    = df['r'] * 100.0
tone = df['tone_mean']
aligned  = pd.concat([r_pct, tone], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma', 'nu']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma, nu = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        s = omega + alpha * r[t-1]**2 + beta * sig2[t-1] + gamma * x[t-1]**2
        sig2[t] = s if s > 1e-8 else 1e-8
    z = r / np.sqrt(sig2)
    ll_t = _t_logpdf(z, nu) - 0.5 * np.log(sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 0.001, 8.0])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun
se    = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr))
extra = [
    f'Estimated degrees of freedom: nu_hat = {theta[4]:.3f}',
    'Note: t-statistic on nu vs 0 is not economically meaningful (nu > 2 by construction).',
]
_report(PARAM_NAMES, theta, se, ll, T_obs,
        persist_alpha_beta=theta[1] + theta[2],
        extra_notes=extra)


# 3. GARCH-X (article growth) -- volume growth of articles matters?


In [5]:
%%capture cap_garchx_art

# === GARCH-X (log article growth)  --  G&P Eq. (4), Student-t innovations =====
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1} + gamma log_art_growth^2_{t-1}
# Using log_art_growth (std≈0.69) instead of raw art_growth (single-day outlier of 61).
print('==== GARCH-X (log article growth)  --  G&P Eq. (4) with x = log_art_growth, Student-t shocks ====')

r_pct    = df['r'] * 100.0
log_artg = df['log_art_growth']
aligned  = pd.concat([r_pct, log_artg], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma', 'nu']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma, nu = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        s = omega + alpha * r[t-1]**2 + beta * sig2[t-1] + gamma * x[t-1]**2
        sig2[t] = s if s > 1e-8 else 1e-8
    z = r / np.sqrt(sig2)
    ll_t = _t_logpdf(z, nu) - 0.5 * np.log(sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 0.01, 8.0])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun
se    = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr))
extra = [
    f'Estimated degrees of freedom: nu_hat = {theta[4]:.3f}',
    'Note: t-statistic on nu vs 0 is not economically meaningful (nu > 2 by construction).',
]
_report(PARAM_NAMES, theta, se, ll, T_obs,
        persist_alpha_beta=theta[1] + theta[2],
        extra_notes=extra)


# 4. GARCHAND (tone) -- tone sign matters?


In [6]:
%%capture cap_garchand_tone

# === GARCHAND (tone)  --  G&P Eq. (5), Student-t innovations ===============
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1}
#             + gamma (1 + d1 * theta) tone^2_{t-1}
# d1 = 1 if tone_{t-1} < 0, else 0;  theta > 0.
print('==== GARCHAND (tone)  --  G&P Eq. (5), Student-t shocks ====')

r_pct    = df['r'] * 100.0
tone = df['tone_mean']
aligned  = pd.concat([r_pct, tone], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma', 'theta', 'nu']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma, theta, nu = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        d1 = 1.0 if x[t-1] < 0 else 0.0
        s = (omega
             + alpha * r[t-1]**2
             + beta  * sig2[t-1]
             + gamma * (1.0 + d1 * theta) * x[t-1]**2)
        sig2[t] = s if s > 1e-8 else 1e-8
    z = r / np.sqrt(sig2)
    ll_t = _t_logpdf(z, nu) - 0.5 * np.log(sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 0.001, 0.5, 8.0])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None),
          (1e-8, None), (2.05, 100.0)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta_hat = opt.x
ll        = -opt.fun
se        = _sandwich_se(theta_hat, neg_ll, per_obs_ll, (r_arr, x_arr))

frac_neg = float((x_arr < 0).mean())
extra = [
    f'Fraction of days with tone_{{t-1}} < 0 (d1 = 1): {frac_neg:.3f}',
    f'Estimated degrees of freedom: nu_hat = {theta_hat[5]:.3f}',
]
_report(PARAM_NAMES, theta_hat, se, ll, T_obs,
        persist_alpha_beta=theta_hat[1] + theta_hat[2],
        extra_notes=extra)


# 5. GARCHAND (article growth) -- volume growth of articles sign matters?


In [7]:
%%capture cap_garchand_art

# === GARCHAND (log article growth)  --  G&P Eq. (6), Student-t innovations ====
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1}
#             + gamma d2 log_art_growth^2_{t-1}
# d2 = 1 if log_art_growth_{t-1} > 0, else 0;  gamma > 0.
print('==== GARCHAND (log article growth)  --  G&P Eq. (6), Student-t shocks ====')

r_pct    = df['r'] * 100.0
log_artg = df['log_art_growth']
aligned  = pd.concat([r_pct, log_artg], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma', 'nu']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma, nu = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        d2 = 1.0 if x[t-1] > 0 else 0.0
        s = (omega
             + alpha * r[t-1]**2
             + beta  * sig2[t-1]
             + gamma * d2 * x[t-1]**2)
        sig2[t] = s if s > 1e-8 else 1e-8
    z = r / np.sqrt(sig2)
    ll_t = _t_logpdf(z, nu) - 0.5 * np.log(sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 0.01, 8.0])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (1e-8, None), (2.05, 100.0)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun
se    = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr))

frac_pos = float((x_arr > 0).mean())
extra = [
    f'Fraction of days with log_art_growth_{{t-1}} > 0 (d2 = 1): {frac_pos:.3f}',
    f'Estimated degrees of freedom: nu_hat = {theta[4]:.3f}',
]
_report(PARAM_NAMES, theta, se, ll, T_obs,
        persist_alpha_beta=theta[1] + theta[2],
        extra_notes=extra)


# 6. GARCHND (tone, article growth) -- volatility regime matters?


In [8]:
%%capture cap_garchnd

# === GARCHND (tone, article growth)  --  G&P Eq. (7), Student-t shocks ====
# News dummy activated only when previous conditional variance exceeds kappa.
# Two annualized vol thresholds calibrated for REMX: 30% and 50%.
print('==== GARCHND (tone, article growth)  --  G&P Eq. (7), Student-t shocks ====')

KAPPA_LOW_ANN  = 30.0
KAPPA_HIGH_ANN = 50.0
def annual_vol_to_daily_var(v_ann_pct):
    return (v_ann_pct / np.sqrt(252.0)) ** 2
KAPPA_LOW  = annual_vol_to_daily_var(KAPPA_LOW_ANN)
KAPPA_HIGH = annual_vol_to_daily_var(KAPPA_HIGH_ANN)
print(f'kappa_low  (vol = {KAPPA_LOW_ANN:>4.1f}% annual) = {KAPPA_LOW:.4f}  (in %^2/day)')
print(f'kappa_high (vol = {KAPPA_HIGH_ANN:>4.1f}% annual) = {KAPPA_HIGH:.4f}  (in %^2/day)')
print()

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma', 'nu']

# Smooth indicator (steep sigmoid) so the LL is differentiable w.r.t. params.
SMOOTH_K = 20.0
def d3_smooth(z):
    return 1.0 / (1.0 + np.exp(-SMOOTH_K * z))

def per_obs_ll(params, r, x, kappa):
    omega, alpha, beta, gamma, nu = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        d3 = d3_smooth(sig2[t-1] - kappa)
        s = (omega
             + alpha * r[t-1]**2
             + beta  * sig2[t-1]
             + gamma * d3 * x[t-1]**2)
        sig2[t] = s if s > 1e-8 else 1e-8
    z = r / np.sqrt(sig2)
    ll_t = _t_logpdf(z, nu) - 0.5 * np.log(sig2)
    return ll_t, sig2

def neg_ll(params, r, x, kappa):
    ll_t, _ = per_obs_ll(params, r, x, kappa)
    return -ll_t.sum()

def _fit_robust(neg_ll_fn, x0, bounds, args):
    opt = minimize(neg_ll_fn, x0, args=args, method='L-BFGS-B',
                   bounds=bounds, options={'maxiter': 5000})
    opt_nm = minimize(neg_ll_fn, opt.x, args=args, method='Nelder-Mead',
                      options={'maxiter': 10000, 'xatol': 1e-7, 'fatol': 1e-7})
    if opt_nm.fun < opt.fun:
        x_proj = np.array([
            min(max(opt_nm.x[i], bounds[i][0] if bounds[i][0] is not None else -np.inf),
                bounds[i][1] if bounds[i][1] is not None else  np.inf)
            for i in range(len(opt_nm.x))
        ])
        opt2 = minimize(neg_ll_fn, x_proj, args=args, method='L-BFGS-B',
                        bounds=bounds, options={'maxiter': 5000})
        if opt2.fun < opt.fun:
            opt = opt2
    return opt

def fit_garchnd(label, x_series, kappa):
    # Pass the raw (un-shifted) exogenous series; the t-1 lag is applied
    # inside per_obs_ll via the x[t-1] indexing, matching G&P Eq. (7).
    r_pct   = df['r'] * 100.0
    aligned = pd.concat([r_pct, x_series], axis=1, keys=['r', 'x']).dropna()
    r_arr = aligned['r'].values
    x_arr = aligned['x'].values
    T_obs = len(r_arr)
    print(f'---- {label} ----')
    print(f'Sample: T = {T_obs} obs '
          f'({aligned.index.min().date()} -> {aligned.index.max().date()})  '
          f'kappa = {kappa:.4f}')

    x0     = np.array([0.086, 0.089, 0.898, 0.01, 8.0])
    bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999),
              (None, None), (2.05, 100.0)]
    opt = _fit_robust(neg_ll, x0, bounds, args=(r_arr, x_arr, kappa))
    theta = opt.x
    ll    = -opt.fun

    se = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr, kappa))

    _, sig2_path = per_obs_ll(theta, r_arr, x_arr, kappa)
    frac_active = float((sig2_path[:-1] >= kappa).mean())
    extra = [
        f'd3 active in-sample (sigma^2_{{t-1}} >= kappa): {frac_active:.3f}',
        f'Estimated degrees of freedom: nu_hat = {theta[4]:.3f}',
    ]

    _report(PARAM_NAMES, theta, se, ll, T_obs,
            persist_alpha_beta=theta[1] + theta[2],
            extra_notes=extra)
    print()

tone     = df['tone_mean']
log_artg = df['log_art_growth']

fit_garchnd('GARCHND  x = Tone             kappa = 30% annual', tone,     KAPPA_LOW)
fit_garchnd('GARCHND  x = Tone             kappa = 50% annual', tone,     KAPPA_HIGH)
fit_garchnd('GARCHND  x = log_art_growth   kappa = 30% annual', log_artg, KAPPA_LOW)
fit_garchnd('GARCHND  x = log_art_growth   kappa = 50% annual', log_artg, KAPPA_HIGH)


# Generate report


In [9]:
from pathlib import Path

REPORTS_DIR = Path("../REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)

sections = [
    ("GARCH(1,1) -- Student-t baseline",        cap_garch),
    ("GARCH-X (tone) -- Student-t",             cap_garchx_tone),
    ("GARCH-X (log article growth) -- Student-t",   cap_garchx_art),
    ("GARCHAND (tone) -- Student-t",                cap_garchand_tone),
    ("GARCHAND (log article growth) -- Student-t",  cap_garchand_art),
    ("GARCHND (tone, log_art_growth) -- Student-t", cap_garchnd),
]

def _banner(title, char='='):
    bar = char * 78
    return f"{bar}\n{title}\n{bar}\n"

out_lines = []
for title, cap in sections:
    out_lines.append(_banner(title, '-'))
    out_lines.append(cap.stdout if cap.stdout else '(no stdout captured)\n')
    out_lines.append("\n")
combined = "".join(out_lines)

(REPORTS_DIR / "03-GP-models-student.txt").write_text(combined)
print(combined)
print(f"Persisted combined Student-t GARCH-family capture -> REPORTS/03-GP-models-student.txt")


------------------------------------------------------------------------------
GARCH(1,1) -- Student-t baseline
------------------------------------------------------------------------------
                          Zero Mean - GARCH Model Results                           
Dep. Variable:                            r   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -5956.23
Distribution:      Standardized Student's t   AIC:                           11920.5
Method:                  Maximum Likelihood   BIC:                           11944.2
                                              No. Observations:                 2765
Date:                      Sat, Jun 06 2026   Df Residuals:                     2765
Time:                              21:29:40   Df Model:                            0
                             Volatility Mode